In [ ]:
%pip install networkx

In [ ]:
import networkx as nx
import numpy as np
import csv

In [ ]:

# ============================================================
# 1. GLOBAL PARAMETERS
# ============================================================

START_YEAR = 2026
END_YEAR = 2040

A_START = 0.0
A_GROWTH = 0.07

TIE_SCALE = 0.15
REGIONAL_TIE = 0.05

NOISE_AMPLITUDE = 0.02
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


# ============================================================
# 2. INDUSTRY RESISTANCE
# ============================================================

base_resistance = {
    "Technology": 0.10,
    "Media": 0.20,
    "Telecom": 0.25,
    "Retail": 0.30,
    "Hospitality": 0.35,
    "Logistics": 0.40,
    "Banking": 0.45,
    "Accounting": 0.50,
    "Manufacturing": 0.50,
    "Insurance": 0.55,
    "Energy": 0.55,
    "Pharma": 0.60,
    "Education": 0.60,
    "Nonprofit": 0.65,
    "Real Estate": 0.65,
    "Construction": 0.70,
    "Agriculture": 0.70,
    "Healthcare": 0.75,
    "Government": 0.85,
    "Legal": 0.90,
}


# ============================================================
# 3. REGIONS AND REGIONAL RESISTANCE SHIFTS
# ============================================================

regions = {
    "Pacific Northwest": -0.049,
    "California": -0.060,
    "Mountain West": 0.076,
    "Southwest": -0.012,
    "Texas": 0.072,
    "Great Plains": 0.077,
    "Midwest": 0.027,
    "Great Lakes": 0.011,
    "Lower South": 0.083,
    "Upper South": 0.027,
    "Mid-Atlantic": -0.048,
    "Northeast": -0.046,
    "Alaska": 0
}

regions = {
    "West": -0.15,
    "Mountain": 0.05,
    "Plains": -0.05,
    "South": .15,
    "North": 0.10
}

regions = {
    "West": -0.15,
    "South": 0.15,
    "North": -0.05
}

regions = {
    "USA": 0
}


# ============================================================
# 4. CREATE INDUSTRY-REGION NODES
# ============================================================

nodes = [
    (industry, region)
    for region in regions
    for industry in base_resistance
]


# Each node receives its own resistance value

resistance = {}

for industry, region in nodes:

    base = base_resistance[industry]
    regional_shift = regions[region]

    noise = rng.uniform(
        -NOISE_AMPLITUDE,
        NOISE_AMPLITUDE
    )

    resistance[(industry, region)] = np.clip(
        base + (2*regional_shift) + noise,
        0.0,
        1.0
    )


# ============================================================
# 5. INDUSTRY PARTNER TIES
# ============================================================

raw_ties = [
    ("Healthcare", "Insurance", 1.0),
    ("Manufacturing", "Logistics", 0.9),
    ("Banking", "Real Estate", 0.8),
    ("Retail", "Logistics", 0.8),
    ("Construction", "Real Estate", 0.8),
    ("Healthcare", "Pharma", 0.7),
    ("Legal", "Real Estate", 0.7),
    ("Healthcare", "Government", 0.6),
    ("Legal", "Government", 0.6),
    ("Accounting", "Banking", 0.6),
    ("Government", "Education", 0.6),
    ("Agriculture", "Logistics", 0.6),
    ("Telecom", "Technology", 0.6),
    ("Insurance", "Banking", 0.5),
    ("Legal", "Banking", 0.5),
    ("Energy", "Manufacturing", 0.5),
    ("Technology", "Media", 0.5),
    ("Insurance", "Legal", 0.4),
    ("Accounting", "Nonprofit", 0.4),
    ("Construction", "Government", 0.4),
    ("Hospitality", "Retail", 0.4),
    ("Pharma", "Logistics", 0.4),
    ("Nonprofit", "Education", 0.4),
    ("Media", "Retail", 0.3),
]


# Scale partner ties

partner_ties = [
    (i, j, strength * TIE_SCALE)
    for i, j, strength in raw_ties
]


# ============================================================
# 6. GEOGRAPHICALLY NEIGHBORING REGIONS
# ============================================================

regional_connections = [
    ("Pacific Northwest", "California"),
    ("Pacific Northwest", "Mountain West"),
    ("California", "Mountain West"),
    ("California", "Southwest"),
    ("Mountain West", "Southwest"),
    ("Mountain West", "Great Plains"),
    ("Southwest", "Texas"),
    ("Southwest", "Great Plains"),
    ("Texas", "Great Plains"),
    ("Texas", "Lower South"),
    ("Great Plains", "Midwest"),
    ("Midwest", "Great Lakes"),
    ("Midwest", "Lower South"),
    ("Great Lakes", "Upper South"),
    ("Great Lakes", "Mid-Atlantic"),
    ("Lower South", "Upper South"),
    ("Upper South", "Mid-Atlantic"),
    ("Mid-Atlantic", "Northeast"),
]

regional_connections = [
    ("West", "Mountain"),
    ("Mountain", "Plains"),
    ("Plains", "South"),
    ("Plains", "North"),
    ("North", "South")
]

regional_connections = [
    ("West", "South"),
    ("West", "North"),
    ("North", "South")
]

regional_connections = [

]

# ============================================================
# 7. BUILD THE FULL NETWORK
# ============================================================

G = nx.DiGraph()

SOURCE = "__SOURCE__"
SINK = "__SINK__"

G.add_nodes_from(nodes)

# Within-region industry relationships

for region in regions:

    for industry1, industry2, J in partner_ties:

        node1 = (industry1, region)
        node2 = (industry2, region)

        G.add_edge(node1, node2, tie=J)
        G.add_edge(node2, node1, tie=J)


# Between-region relationships
# Same industry in neighboring regions

for region1, region2 in regional_connections:

    for industry in base_resistance:

        node1 = (industry, region1)
        node2 = (industry, region2)

        G.add_edge(
            node1, node2,
            tie=REGIONAL_TIE
        )

        G.add_edge(
            node2, node1,
            tie=REGIONAL_TIE
        )


In [ ]:

# ============================================================
# EXTRACT NETWORKX GRAPH FOR QAOA
# ============================================================

# Preserve the original node ordering.
qaoa_nodes = list(G.nodes())
N = len(qaoa_nodes)

node_index = {
    node: i for i, node in enumerate(qaoa_nodes)
}

# Extract each undirected relationship exactly once.
edges = []

for u, v, data in G.edges(data=True):

    # Only keep one direction of each reciprocal edge.
    if node_index[u] < node_index[v]:

        J = data["tie"]

        edges.append((
            node_index[u],
            node_index[v],
            J
        ))

# Extract resistance from your existing base_resistance dictionary.
# Nodes are tuples: (industry, region).
resistance = np.array([
    base_resistance[industry]
    for industry, region in qaoa_nodes
])

print("Number of nodes:", N)
print("Number of Ising interactions:", len(edges))



Number of nodes: 20
Number of Ising interactions: 24


In [ ]:
%pip install "openquantum-sdk[pennylane]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openquantum-sdk-pennylane]


In [ ]:
import numpy as onp
from collections import Counter

# ============================================================
# QAOA CONFIGURATION PARAMETERS
# ============================================================
p = 1                      # Number of QAOA layers
shots = 1000               # Number of shots for sampling
exact_limit = 12           # Limit for classical exact solution to prevent memory issues
learning_rate = 0.01       # Optimization learning rate
iterations = 10            # Number of optimization steps per year (kept low for demo/quick execution)

# Years and AI Pressure calculations
years = list(range(START_YEAR, END_YEAR + 1))
A_by_year = {
    year: A_START + A_GROWTH * (year - START_YEAR)
    for year in years
}

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp

# ============================================================
# 4. DEFINE THE ISING ENERGY
# ============================================================

# E = -sum(J_ij*s_i*s_j) - sum((A-r_i)*s_i)
#
# s_i = +1: digital
# s_i = -1: paper

def calculate_energy(state, A):

    energy = 0.0

    # Interactions between nodes.
    for i, j, J in edges:
        energy -= J * state[i] * state[j]

    # Individual node contributions.
    for i in range(N):
        energy -= (A - resistance[i]) * state[i]

    return float(energy)


# ============================================================
# 5. CONSTRUCT THE ISING HAMILTONIAN
# ============================================================

def construct_hamiltonian(A):

    coefficients = []
    observables = []

    # Coupling terms.
    for i, j, J in edges:

        coefficients.append(-J)

        observables.append(
            qml.PauliZ(i) @ qml.PauliZ(j)
        )

    # Individual node terms.
    for i in range(N):

        h = A - resistance[i]

        coefficients.append(-h)
        observables.append(qml.PauliZ(i))

    return qml.Hamiltonian(coefficients, observables)


# ============================================================
# 6. OPTIONAL EXACT CLASSICAL SOLUTION
# ============================================================

def exact_classical_solution(A):

    if N > exact_limit:
        return None, None

    best_energy = float("inf")
    best_state = None

    for state in product([-1, 1], repeat=N):

        energy = calculate_energy(state, A)

        if energy < best_energy:
            best_energy = energy
            best_state = state

    return best_energy, best_state


# ============================================================
# 7. CREATE QAOA CIRCUIT
# ============================================================

# Mixer Hamiltonian:
# H_M = sum_i X_i

mixer_h = qml.Hamiltonian(
    [1.0] * N,
    [qml.PauliX(i) for i in range(N)]
)


def create_qaoa_circuit(cost_h):

    def circuit(params):

        # Initial equal superposition.
        for i in range(N):
            qml.Hadamard(wires=i)

        # Alternating cost and mixer layers.
        for layer in range(p):

            qml.qaoa.cost_layer(
                params[0, layer],
                cost_h
            )

            qml.qaoa.mixer_layer(
                params[1, layer],
                mixer_h
            )

    return circuit


# ============================================================
# 8. SIMULATION DEVICES
# ============================================================

# Analytic simulator for parameter optimization.
# This avoids sampling noise during optimization.
optimization_device = qml.device(
    "default.qubit",
    wires=N
)

# Finite-shot simulator for final measurements.
sampling_device = qml.device(
    "openquantum.device",
    wires=N,
    shots=shots,
    backend="rigetti:cepheus-1-108q",
    client_id = "s_7a87ad30a09e4cf8bfb1a8ebd643d3ed",
    client_secret = "e76fc3e0c7f7620ea1d4c76c63cbc6e8821a4dc29110a2df0b90343334d61065"
)


# ============================================================
# 9. RUN QAOA FOR ONE YEAR
# ============================================================

def run_qaoa_year(A, initial_params=None):

    cost_h = construct_hamiltonian(A)

    circuit = create_qaoa_circuit(cost_h)

    # Expected energy of the circuit.
    @qml.qnode(optimization_device)
    def cost_function(params):

        circuit(params)

        return qml.expval(cost_h)

    # Initialize parameters using PennyLane's numpy wrapper
    if initial_params is None:

        params = pnp.array(
            [
                onp.random.uniform(0, onp.pi, p),
                onp.random.uniform(0, onp.pi, p)
            ],
            requires_grad=True
        )

    else:

        params = pnp.array(
            initial_params,
            requires_grad=True
        )

    # Optimize gamma and beta.
    optimizer = qml.AdamOptimizer(
        stepsize=learning_rate
    )

    for step in range(iterations):

        params = optimizer.step(
            cost_function,
            params
        )

        if step % 50 == 0:
            print(
                f"    Iteration {step}: "
                f"Energy = {float(cost_function(params)):.6f}"
            )

    expected_energy = float(cost_function(params))

    # Sample the optimized circuit.
    @qml.qnode(sampling_device)
    def sample_circuit(params):

        circuit(params)

        return qml.sample(wires=range(N))

    samples = sample_circuit(params)

    # Convert bitstrings to spin configurations.
    # 0 -> +1 -> digital
    # 1 -> -1 -> paper

    counts = Counter()

    for bitstring in samples:

        state = tuple(
            1 - 2 * int(bit)
            for bit in bitstring
        )

        counts[state] += 1

    # Rank sampled configurations by energy.
    ranked_results = []

    for state, count in counts.items():

        energy = calculate_energy(state, A)

        ranked_results.append(
            (energy, state, count)
        )

    ranked_results.sort(key=lambda x: x[0])

    best_energy, best_state, best_count = ranked_results[0]

    return {
        "params": onp.array(params),
        "expected_energy": expected_energy,
        "best_energy": best_energy,
        "best_state": best_state,
        "best_count": best_count,
        "ranked_results": ranked_results
    }


# ============================================================
# 10. RUN ALL YEARS
# ============================================================

yearly_results = {}

# Previous year's optimized parameters.
previous_params = None

for year in years:

    A = A_by_year[year]

    print("\n" + "=" * 55)
    print(f"YEAR: {year}")
    print(f"AI PRESSURE: {A}")
    print("=" * 55)

    result = run_qaoa_year(
        A,
        initial_params=previous_params
    )

    yearly_results[year] = result

    # Warm-start the next year.
    previous_params = result["params"]

    print("\nBest sampled energy:", result["best_energy"])
    print("Expected energy:", result["expected_energy"])
    print("Best sample frequency:", result["best_count"], "/", shots)

    # Display the best configuration.
    print("\nBest configuration:")

    for i, state in enumerate(result["best_state"]):

        status = "DIGITAL" if state == 1 else "PAPER"

        print(f"{qaoa_nodes[i]}: {status}")


# ============================================================
# 11. SAVE YEARLY SUMMARY
# ============================================================

summary_file = "qaoa_yearly_summary.csv"

with open(summary_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Year",
        "AI_pressure",
        "Expected_energy",
        "Best_sampled_energy",
        "Best_sample_count",
        "Best_sample_probability",
        "Exact_energy",
        "Energy_gap"
    ])

    for year in years:

        result = yearly_results[year]

        exact_energy, exact_state = exact_classical_solution(
            A_by_year[year]
        )

        if exact_energy is None:
            energy_gap = None
        else:
            energy_gap = result["best_energy"] - exact_energy

        writer.writerow([
            year,
            A_by_year[year],
            result["expected_energy"],
            result["best_energy"],
            result["best_count"],
            result["best_count"] / shots,
            exact_energy,
            energy_gap
        ])


# ============================================================
# 12. SAVE NODE STATES FOR EVERY YEAR
# ============================================================

state_file = "qaoa_node_states.csv"

with open(state_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Year"
    ] + [
        str(node) for node in qaoa_nodes
    ])

    for year in years:

        state = yearly_results[year]["best_state"]

        writer.writerow([
            year
        ] + list(state))


# ============================================================
# 13. CALCULATE FIRST DIGITAL YEAR
# ============================================================

flip_file = "qaoa_flip_years.csv"

with open(flip_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Industry",
        "Region",
        "First_Digital_Year"
    ])

    for i, node in enumerate(qaoa_nodes):

        industry, region = node

        flip_year = None

        for year in years:

            state = yearly_results[year]["best_state"]

            if state[i] == 1:

                flip_year = year
                break

        writer.writerow([
            industry,
            region,
            flip_year
        ])


# ============================================================
# 14. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 55)
print("QAOA SIMULATION COMPLETE")
print("=" * 55)

print("Results saved:")
print("  ", summary_file)
print("  ", state_file)
print("  ", flip_file)

print("\nFirst digital year:")

for i, node in enumerate(qaoa_nodes):

    flip_year = None

    for year in years:

        if yearly_results[year]["best_state"][i] == 1:
            flip_year = year
            break

    print(f"{node}: {flip_year}")

/usr/local/lib/python3.13/dist-packages/pennylane/devices/device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(



YEAR: 2026
AI PRESSURE: 0.0
    Iteration 0: Energy = 2.557009
[Open Quantum] Batch: 1 circuit(s) × 1 credit(s) each = 1 credit(s) total
[Open Quantum] Proceed? [y/N] y

Note: You have selected the Public Plan. Any publications or public work resulting from this job must provide proper attribution to Open Quantum. Citation guidelines can be found at www.openquantum.com/citation.


Best sampled energy: -9.630000000000003
Expected energy: 1.2614602534228454
Best sample frequency: 1 / 1000

Best configuration:
('Technology', 'USA'): DIGITAL
('Media', 'USA'): PAPER
('Telecom', 'USA'): PAPER
('Retail', 'USA'): PAPER
('Hospitality', 'USA'): PAPER
('Logistics', 'USA'): DIGITAL
('Banking', 'USA'): PAPER
('Accounting', 'USA'): PAPER
('Manufacturing', 'USA'): DIGITAL
('Insurance', 'USA'): PAPER
('Energy', 'USA'): PAPER
('Pharma', 'USA'): PAPER
('Education', 'USA'): PAPER
('Nonprofit', 'USA'): PAPER
('Real Estate', 'USA'): PAPER
('Construction', 'USA'): PAPER
('Agriculture', 'USA'): PAPER
('Heal